# TCN Evaluation Analysis Notebook

This notebook is the clean post-training evaluation workflow for the final `Run21` analysis.

It is organized around three evaluation blocks:
1. deterministic checkpoint selection across 1, 2, 3, and 4 year horizons
2. stratified stochastic confirmation using both regime-stratified and year-stratified windows
3. benchmark comparison versus equal-weight and SPY across the same horizons

All outputs are written into a timestamped analysis session directory with stable subfolders for metrics, tracks, and visualization-ready artifacts.


## 1) Setup
Use this notebook after `Run21` finishes. Keep the training config frozen. Only evaluation settings should change here.


In [ ]:
import copy
import importlib.util
import json
import os
import re
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path

from IPython.display import display

# GitHub/runtime bootstrap, aligned with the training notebook.
GIT_REPO_URL = "https://github.com/Dave-DKings/tcn_tape_vectorized_version.git"
GIT_BRANCH = "feature/run17-film-drive-org-20260318"
CLONE_IF_MISSING = True
CLONE_PARENT_DIR = Path('/content')
CLONE_DIR_NAME = 'tcn_tape_vectorized_version_clean'
EVAL_BRANCH = None
INSTALL_REQUIREMENTS = False
AUTO_INSTALL_MISSING_REQUIREMENTS = True

CRITICAL_RUNTIME_MODULES = {
    'pandas_ta_classic': 'pandas-ta-classic>=0.3.59',
    'fredapi': 'fredapi>=0.5.1',
    'yfinance': 'yfinance>=0.2.38',
}


def run(cmd):
    print('+', ' '.join(map(str, cmd)))
    subprocess.run(cmd, check=True)


def missing_runtime_requirements() -> list[str]:
    missing = []
    for module_name, requirement in CRITICAL_RUNTIME_MODULES.items():
        if importlib.util.find_spec(module_name) is None:
            missing.append(requirement)
    return missing


def normalize_github_url(url: str | None) -> str | None:
    if not url:
        return url
    url = str(url).strip()
    if url.startswith('git@github.com:'):
        repo = url[len('git@github.com:'):]
        if repo.endswith('.git'):
            repo = repo[:-4]
        return f'https://github.com/{repo}.git'
    return url


def safe_cwd() -> Path | None:
    try:
        return Path.cwd().resolve()
    except FileNotFoundError:
        return None


def safe_resolve(path_like) -> Path | None:
    p = Path(path_like)
    try:
        if p.is_absolute():
            return p.resolve()
    except Exception:
        return None
    cwd = safe_cwd()
    if cwd is None:
        return None
    try:
        return (cwd / p).resolve()
    except Exception:
        return None


def find_repo_root() -> Path | None:
    candidate_roots = []
    seen = set()

    cwd = safe_cwd()
    if cwd is not None:
        for p in [cwd, *cwd.parents]:
            rp = safe_resolve(p)
            if rp is not None and rp not in seen:
                seen.add(rp)
                candidate_roots.append(rp)

    for p in [
        Path(globals().get('EVAL_REPO_DIR', CLONE_PARENT_DIR / CLONE_DIR_NAME)),
        CLONE_PARENT_DIR / CLONE_DIR_NAME,
        Path('/content/tcn_tape_vectorized_version_clean'),
        Path('/content/tcn_tape_vectorized_version'),
        Path('/content/repo'),
        Path('/content/project'),
        Path('/content/drive/MyDrive/tcn_tape_vectorized_version_clean'),
        Path('/mnt/c/Users/Owner/tcn_tape_vectorized_version_clean'),
    ]:
        rp = safe_resolve(p)
        if rp is not None and rp not in seen:
            seen.add(rp)
            candidate_roots.append(rp)

    for p in candidate_roots:
        if (p / '.git').exists() and (p / 'src').exists() and (p / 'tcn_evaluation_analysis.ipynb').exists():
            return p
        if (p / 'src' / 'config.py').exists():
            return p
    return None


REPO_ROOT = find_repo_root()
GIT_REPO_URL = normalize_github_url(GIT_REPO_URL)

if REPO_ROOT is None and CLONE_IF_MISSING:
    if not GIT_REPO_URL:
        raise FileNotFoundError(
            'Repo root not found and GIT_REPO_URL is not set. '
            'Set GIT_REPO_URL or EVAL_REPO_DIR before running this notebook.'
        )
    CLONE_PARENT_DIR.mkdir(parents=True, exist_ok=True)
    clone_target = CLONE_PARENT_DIR / CLONE_DIR_NAME
    if clone_target.exists() and not (clone_target / '.git').exists():
        print(f'[WARN] Removing partial non-git clone target: {clone_target}')
        shutil.rmtree(clone_target, ignore_errors=True)
    if not clone_target.exists():
        clone_cmd = ['git', 'clone', GIT_REPO_URL, str(clone_target)]
        print('+', ' '.join(map(str, clone_cmd)))
        clone_proc = subprocess.run(clone_cmd, text=True, capture_output=True)
        if clone_proc.stdout:
            print(clone_proc.stdout, end='')
        if clone_proc.returncode != 0:
            stderr = (clone_proc.stderr or '').strip()
            raise RuntimeError(f'git clone failed: {stderr}')
    REPO_ROOT = clone_target.resolve()

if REPO_ROOT is None:
    raise FileNotFoundError(
        'Could not locate repo root containing src/config.py. '
        'Set EVAL_REPO_DIR manually before running this notebook.'
    )

if GIT_BRANCH and (REPO_ROOT / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin'], check=False)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', GIT_BRANCH], check=False)

if EVAL_BRANCH and (REPO_ROOT / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', EVAL_BRANCH], check=False)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

requirements_file = REPO_ROOT / 'requirements.txt'
missing_requirements = missing_runtime_requirements()
should_install = INSTALL_REQUIREMENTS or (AUTO_INSTALL_MISSING_REQUIREMENTS and bool(missing_requirements))
if should_install:
    run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'])
    if INSTALL_REQUIREMENTS and requirements_file.exists():
        run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_file)])
    elif missing_requirements:
        run([sys.executable, '-m', 'pip', 'install', *missing_requirements])

EVAL_REPO_DIR = str(REPO_ROOT)
print('EVAL_REPO_DIR:', EVAL_REPO_DIR)
print('Missing critical requirements before install:', missing_requirements)
print('Dependency install executed:', should_install)

import numpy as np
import pandas as pd

from src.config import build_run21_config
from src.notebook_helpers.tcn_phase1 import (
    prepare_phase1_dataset,
    create_experiment6_result_stub,
    evaluate_experiment6_checkpoint,
    load_training_metadata_into_config,
    compare_agent_vs_baseline,
    Phase1Dataset,
)


In [ ]:
RUN_ID = "run21"
EVAL_RANDOM_SEED = 42
EVAL_FORCE_TEST_START_DATE = "2020-01-01"
EVAL_DETERMINISTIC_MODE = "mean"
EVAL_STOCHASTIC_MODE = "sample"

# Horizons used throughout the analysis.
HORIZON_YEARS = [1, 2, 3, 4]
HORIZON_DAYS = {year: 252 * year for year in HORIZON_YEARS}

# Deterministic checkpoint sweep.
EVAL_SWEEP_START_OFFSETS = [0, 63, 126]
EVAL_TOP_HW = 40
EVAL_TOP_PERIODIC = 6
EVAL_INCLUDE_ROOT = True
EVAL_INCLUDE_RARE = False

# Checkpoint basket control.
USE_MANUAL_CHECKPOINT_EPISODES = True
MANUAL_CHECKPOINT_EPISODES = [528, 516, 511, 496, 489]

# Stochastic confirmations.
STOCH_RUNS_PER_WINDOW = 16
STOCH_EPISODE_LIMIT = 252
YEAR_WINDOWS_PER_YEAR = 3
YEAR_WINDOW_MIN_GAP_DAYS = 42

# Artifact controls.
EVAL_SAVE_LOGS = True
EVAL_SAVE_ARTIFACTS = True
CAPTURE_RECENT_ARTIFACT_HOURS = 12

# Optional Drive restore bootstrap for saved run outputs.
AUTO_RESTORE_RESULTS_FROM_DRIVE = True
FORCE_RESTORE_RESULTS = False
RUN_RESULTS_ZIP_PATH = Path(
    globals().get(
        'RUN_RESULTS_ZIP_PATH',
        '/content/drive/MyDrive/tcn_tape_vectorized_runs/run21/tcn_tape_vectorized_run21.zip',
    )
)
EVAL_RESTORE_DIR = Path(globals().get('EVAL_RESTORE_DIR', '/content/eval_restore'))
repo_dir = Path(globals().get('EVAL_REPO_DIR', str(REPO_ROOT)))
restore_dir = EVAL_RESTORE_DIR


def _has_metadata(results_root: Path) -> bool:
    logs_dir = results_root / 'logs'
    return logs_dir.exists() and any(logs_dir.glob('*_metadata.json'))


def _running_in_colab() -> bool:
    return importlib.util.find_spec('google.colab') is not None


def _maybe_mount_drive() -> None:
    if not _running_in_colab():
        return
    drive_root = Path('/content/drive/MyDrive')
    if drive_root.exists():
        return
    from google.colab import drive
    drive.mount('/content/drive')


def _restore_results_from_drive_if_needed() -> None:
    if not AUTO_RESTORE_RESULTS_FROM_DRIVE:
        return
    if _running_in_colab():
        _maybe_mount_drive()
    if not RUN_RESULTS_ZIP_PATH.exists():
        print('[INFO] Run results zip not found on Drive:', RUN_RESULTS_ZIP_PATH)
        return
    current_root = restore_dir / 'tcn_fusion_results'
    if current_root.exists() and _has_metadata(current_root) and not FORCE_RESTORE_RESULTS:
        print('[OK] Existing restored results found:', current_root)
        return
    if restore_dir.exists():
        shutil.rmtree(restore_dir, ignore_errors=True)
    restore_dir.mkdir(parents=True, exist_ok=True)
    import zipfile
    with zipfile.ZipFile(RUN_RESULTS_ZIP_PATH, 'r') as zf:
        zf.extractall(restore_dir)
    print('[OK] Restored results zip to:', restore_dir)


def _find_named_dirs(base: Path, dir_name: str) -> list[Path]:
    if not base.exists():
        return []
    matches = []
    direct = base / dir_name
    if direct.exists() and direct.is_dir():
        matches.append(direct)
    try:
        for p in base.rglob(dir_name):
            if p.is_dir() and p not in matches:
                matches.append(p)
    except Exception:
        pass
    return matches


_restore_results_from_drive_if_needed()

results_candidates = []
for root in [restore_dir, repo_dir]:
    for candidate in _find_named_dirs(root, 'tcn_fusion_results'):
        if candidate not in results_candidates:
            results_candidates.append(candidate)

metadata_candidates = [p for p in results_candidates if _has_metadata(p)]
if metadata_candidates:
    EVAL_RESULTS_ROOT = sorted(
        metadata_candidates,
        key=lambda p: max((f.stat().st_mtime for f in (p / 'logs').glob('*_metadata.json')), default=0),
        reverse=True,
    )[0]
elif results_candidates:
    EVAL_RESULTS_ROOT = results_candidates[0]
else:
    EVAL_RESULTS_ROOT = restore_dir / 'tcn_fusion_results'

prep_candidates = []
for root in [restore_dir, EVAL_RESULTS_ROOT.parent, repo_dir]:
    for candidate in _find_named_dirs(root, 'data_exports'):
        if candidate not in prep_candidates:
            prep_candidates.append(candidate)
EVAL_PREP_ARTIFACTS_DIR = next((p for p in prep_candidates if p.exists()), repo_dir / 'data_exports')

SESSION_TAG = datetime.utcnow().strftime('run21_eval_%Y%m%d_%H%M%S_utc')
EVAL_ANALYSIS_ROOT = EVAL_RESULTS_ROOT / 'analysis' / RUN_ID / SESSION_TAG
EVAL_DIRS = {
    'metadata': EVAL_ANALYSIS_ROOT / '00_metadata',
    'checkpoint_selection': EVAL_ANALYSIS_ROOT / '01_checkpoint_selection',
    'stratified_regime': EVAL_ANALYSIS_ROOT / '02_stratified_stochastic' / 'regime',
    'stratified_year': EVAL_ANALYSIS_ROOT / '02_stratified_stochastic' / 'year',
    'benchmarks': EVAL_ANALYSIS_ROOT / '03_benchmarks',
    'tracks': EVAL_ANALYSIS_ROOT / '04_tracks',
    'visuals': EVAL_ANALYSIS_ROOT / '05_visualizations',
    'manifests': EVAL_ANALYSIS_ROOT / '06_manifests',
}
for p in EVAL_DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

print('RUN_ID:', RUN_ID)
print('EVAL_RESULTS_ROOT:', EVAL_RESULTS_ROOT)
print('EVAL_PREP_ARTIFACTS_DIR:', EVAL_PREP_ARTIFACTS_DIR)
print('EVAL_ANALYSIS_ROOT:', EVAL_ANALYSIS_ROOT)
print('Horizons:', HORIZON_DAYS)


## 2) Build Evaluation Dataset and Metadata-Aligned Config
The evaluation config should come from the saved `Run21` metadata, not from ad hoc overrides.


In [ ]:
eval_config = build_run21_config('phase1')

if EVAL_FORCE_TEST_START_DATE:
    forced_start = pd.Timestamp(EVAL_FORCE_TEST_START_DATE)
    eval_config['TRAIN_TEST_SPLIT_DATE'] = str((forced_start - pd.Timedelta(days=1)).date())


def _find_metadata_files(search_roots: list[Path]) -> list[Path]:
    files = []
    for root in search_roots:
        logs_dir = root / 'logs'
        if logs_dir.exists():
            files.extend(logs_dir.glob('*_metadata.json'))
    unique = []
    seen = set()
    for p in sorted(files, key=lambda p: p.stat().st_mtime, reverse=True):
        rp = p.resolve()
        if rp not in seen:
            seen.add(rp)
            unique.append(rp)
    return unique


metadata_search_roots = []
for root in [EVAL_RESULTS_ROOT, restore_dir / 'tcn_fusion_results', repo_dir / 'tcn_fusion_results']:
    if root not in metadata_search_roots:
        metadata_search_roots.append(root)

meta_files = _find_metadata_files(metadata_search_roots)
if not meta_files:
    searched = '\n'.join(str(root / 'logs') for root in metadata_search_roots)
    raise FileNotFoundError(f'No metadata JSON found. Searched:\n{searched}')

matched_meta = []
for meta_path in meta_files:
    try:
        meta = json.loads(meta_path.read_text(encoding='utf-8'))
        training = meta.get('Training_Settings', {}) or {}
        if str(training.get('high_watermark_checkpoint_subdir', '')).strip() == 'high_watermark_checkpoints_run21':
            matched_meta.append(meta_path)
    except Exception:
        pass

EVAL_METADATA_PATH = matched_meta[0] if matched_meta else meta_files[0]
print('Using metadata:', EVAL_METADATA_PATH)
load_training_metadata_into_config(EVAL_METADATA_PATH, eval_config)

EVAL_HW_SUBDIR = str(eval_config.get('training_params', {}).get('high_watermark_checkpoint_subdir', 'high_watermark_checkpoints_run21'))
EVAL_STEP_SUBDIR = str(eval_config.get('training_params', {}).get('step_sharpe_checkpoint_subdir', 'step_sharpe_checkpoints_run21'))

prep_dir = Path(EVAL_PREP_ARTIFACTS_DIR)
if not prep_dir.exists():
    print('[WARN] Preparation artifacts directory not found:', prep_dir)

eval_phase1_data = prepare_phase1_dataset(
    eval_config,
    force_download=False,
    save_preparation_artifacts=False,
    preparation_artifacts_dir=str(prep_dir) if prep_dir.exists() else None,
)

print('Train shape:', eval_phase1_data.train_df.shape)
print('Test shape:', eval_phase1_data.test_df.shape)
print('Tickers:', eval_config['ASSET_TICKERS'])

(EVAL_DIRS['metadata'] / 'selected_metadata_path.txt').write_text(str(EVAL_METADATA_PATH), encoding='utf-8')
(EVAL_DIRS['metadata'] / 'selected_results_root.txt').write_text(str(EVAL_RESULTS_ROOT), encoding='utf-8')
(EVAL_DIRS['metadata'] / 'selected_prep_artifacts_dir.txt').write_text(str(prep_dir), encoding='utf-8')
(EVAL_DIRS['metadata'] / 'eval_config_snapshot.json').write_text(
    json.dumps(eval_config, indent=2, default=str), encoding='utf-8'
)


## 3) Reusable Helpers
These helpers keep the analysis blocks concise and enforce consistent artifact saving.


In [ ]:
def save_df(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    return path


def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, default=str), encoding='utf-8')
    return path


def capture_recent_eval_artifacts(target_dir: Path, hours_back: int = CAPTURE_RECENT_ARTIFACT_HOURS):
    cutoff = datetime.utcnow().timestamp() - hours_back * 3600
    copied = []
    for src_dir in [EVAL_RESULTS_ROOT / 'logs', EVAL_RESULTS_ROOT / 'artifacts', EVAL_RESULTS_ROOT / 'eval']:
        if not src_dir.exists():
            continue
        for p in src_dir.rglob('*'):
            if not p.is_file():
                continue
            if p.stat().st_mtime < cutoff:
                continue
            rel = p.relative_to(EVAL_RESULTS_ROOT)
            dst = target_dir / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, dst)
            copied.append(str(rel))
    return copied


def discover_checkpoint_pairs(
    results_root: Path,
    *,
    high_watermark_subdir: str | None = None,
    step_sharpe_subdir: str | None = None,
    include_root: bool = True,
) -> pd.DataFrame:
    rows = []
    search_dirs = []
    if high_watermark_subdir:
        search_dirs.append((results_root / high_watermark_subdir, 'high_watermark'))
    if step_sharpe_subdir:
        search_dirs.append((results_root / step_sharpe_subdir, 'periodic_step'))
    if include_root:
        search_dirs.append((results_root, 'root'))

    for base_dir, kind in search_dirs:
        if not base_dir.exists():
            continue
        pattern = '*_actor.weights.h5' if kind == 'root' else '**/*_actor.weights.h5'
        for actor in sorted(base_dir.glob(pattern)):
            prefix = str(actor).replace('_actor.weights.h5', '')
            critic = Path(prefix + '_critic.weights.h5')
            if not critic.exists():
                continue
            name = actor.name
            m_sh = re.search(r'_sh([pm])(\d+)p(\d+)', name)
            sharpe_tag = None
            if m_sh:
                sign = 1.0 if m_sh.group(1) == 'p' else -1.0
                sharpe_tag = sign * float(f"{m_sh.group(2)}.{m_sh.group(3)}")
            m_ep = re.search(r'_ep(\d+)', name)
            m_st = re.search(r'_step(\d+)', name)
            rows.append({
                'checkpoint_prefix': prefix,
                'actor_path': str(actor),
                'critic_path': str(critic),
                'checkpoint_kind': kind,
                'episode': int(m_ep.group(1)) if m_ep else np.nan,
                'step': int(m_st.group(1)) if m_st else np.nan,
                'sharpe_tag': sharpe_tag,
                'mtime': actor.stat().st_mtime,
            })
    if not rows:
        raise RuntimeError(f'No valid actor+critic checkpoints under {results_root}')
    return pd.DataFrame(rows).sort_values(
        ['checkpoint_kind', 'episode', 'step', 'mtime'],
        ascending=[True, True, True, False],
    )


def build_checkpoint_basket(df_ckpt: pd.DataFrame) -> pd.DataFrame:
    out = []
    hw = df_ckpt[df_ckpt['checkpoint_kind'] == 'high_watermark'].copy()
    if not hw.empty:
        out.append(hw.sort_values(['sharpe_tag', 'mtime'], ascending=[False, False]).head(EVAL_TOP_HW))
    periodic = df_ckpt[df_ckpt['checkpoint_kind'] == 'periodic_step'].copy()
    if not periodic.empty:
        out.append(periodic.sort_values(['step', 'mtime'], ascending=[False, False]).head(EVAL_TOP_PERIODIC))
    if EVAL_INCLUDE_ROOT:
        root = df_ckpt[df_ckpt['checkpoint_kind'] == 'root'].copy()
        if not root.empty:
            out.append(root.sort_values('mtime', ascending=False).head(1))
    if not out:
        raise RuntimeError('Checkpoint basket is empty')
    basket = pd.concat(out, ignore_index=True).drop_duplicates(subset=['checkpoint_prefix']).reset_index(drop=True)
    return basket


def eval_run_one_checkpoint(eval_cfg, phase1_data, ckpt_prefix, *, seed, num_eval_runs, horizon_days, save_logs, save_artifacts):
    stub = create_experiment6_result_stub(
        random_seed=seed,
        use_covariance=True,
        architecture=eval_cfg['agent_params']['actor_critic_type'],
        checkpoint_path=ckpt_prefix,
        agent_config=copy.deepcopy(eval_cfg['agent_params']),
        base_agent_params=None,
    )
    return evaluate_experiment6_checkpoint(
        experiment6=stub,
        phase1_data=phase1_data,
        config=eval_cfg,
        random_seed=seed,
        checkpoint_path_override=ckpt_prefix,
        deterministic_eval_mode=EVAL_DETERMINISTIC_MODE,
        num_eval_runs=int(num_eval_runs),
        stochastic_eval_mode=EVAL_STOCHASTIC_MODE,
        stochastic_episode_length_limit=int(min(STOCH_EPISODE_LIMIT, horizon_days)),
        save_eval_logs=bool(save_logs),
        save_eval_artifacts=bool(save_artifacts),
    )


def make_phase1_slice(base_phase1: Phase1Dataset, start_offset: int, horizon_days: int):
    test_df = base_phase1.test_df.copy()
    test_df['Date'] = pd.to_datetime(test_df['Date'])
    unique_dates = pd.Series(test_df['Date'].dropna().unique()).sort_values().reset_index(drop=True)
    if start_offset >= len(unique_dates):
        return None, None
    end_idx = min(len(unique_dates), int(start_offset) + int(horizon_days))
    win_dates = unique_dates.iloc[int(start_offset):end_idx]
    if len(win_dates) < max(60, int(horizon_days * 0.5)):
        return None, None
    d0 = pd.to_datetime(win_dates.iloc[0])
    d1 = pd.to_datetime(win_dates.iloc[-1])
    sliced_df = test_df[(test_df['Date'] >= d0) & (test_df['Date'] <= d1)].copy()
    if sliced_df.empty:
        return None, None
    phase1_slice = copy.deepcopy(base_phase1)
    phase1_slice.test_df = sliced_df
    phase1_slice.test_start_date = d0
    return phase1_slice, {'start_date': str(d0.date()), 'end_date': str(d1.date()), 'n_days': int(len(win_dates))}


## 4) Checkpoint Discovery
Select the candidate basket once. The deterministic sweep uses this basket only.


In [ ]:
print('High-watermark subdir:', EVAL_HW_SUBDIR)
print('Step-sharpe subdir:', EVAL_STEP_SUBDIR)
print('Manual checkpoint mode:', USE_MANUAL_CHECKPOINT_EPISODES)
print('Manual episodes:', MANUAL_CHECKPOINT_EPISODES if USE_MANUAL_CHECKPOINT_EPISODES else [])


In [ ]:
eval_ckpt_df = discover_checkpoint_pairs(
    EVAL_RESULTS_ROOT,
    high_watermark_subdir=EVAL_HW_SUBDIR,
    step_sharpe_subdir=EVAL_STEP_SUBDIR,
    include_root=EVAL_INCLUDE_ROOT,
)

if USE_MANUAL_CHECKPOINT_EPISODES:
    manual_basket = (
        eval_ckpt_df[
            (eval_ckpt_df['checkpoint_kind'] == 'high_watermark')
            & (eval_ckpt_df['episode'].isin(MANUAL_CHECKPOINT_EPISODES))
        ]
        .copy()
    )
    manual_basket['episode'] = manual_basket['episode'].astype(int)
    eval_ckpt_basket = (
        manual_basket
        .sort_values('episode', ascending=False)
        .drop_duplicates(subset=['checkpoint_prefix'])
        .reset_index(drop=True)
    )
    missing = sorted(set(MANUAL_CHECKPOINT_EPISODES) - set(eval_ckpt_basket['episode'].tolist()))
    if missing:
        raise RuntimeError(f'Missing requested checkpoints: {missing}')
else:
    eval_ckpt_basket = build_checkpoint_basket(eval_ckpt_df)

if eval_ckpt_basket.empty:
    raise RuntimeError('No checkpoints selected for evaluation.')

display(eval_ckpt_basket[['checkpoint_kind', 'episode', 'step', 'sharpe_tag', 'checkpoint_prefix']])
save_df(eval_ckpt_basket, EVAL_DIRS['checkpoint_selection'] / 'checkpoint_basket.csv')


## 5) Deterministic Checkpoint Selection
This block ranks checkpoints deterministically across 1, 2, 3, and 4 year horizons.

The purpose is selection, not stochastic robustness. Keep it deterministic only.


In [ ]:
det_records = []
for _, row in eval_ckpt_basket.iterrows():
    ckpt_prefix = row['checkpoint_prefix']
    ckpt_label = f"{row['checkpoint_kind']}__ep{int(row['episode']) if pd.notna(row['episode']) else -1:04d}"
    for years, horizon_days in HORIZON_DAYS.items():
        for start_offset in EVAL_SWEEP_START_OFFSETS:
            phase1_slice, meta = make_phase1_slice(eval_phase1_data, start_offset, horizon_days)
            if phase1_slice is None:
                continue
            seed = int(EVAL_RANDOM_SEED + years * 1000 + start_offset)
            try:
                ev = eval_run_one_checkpoint(
                    eval_config,
                    phase1_slice,
                    ckpt_prefix,
                    seed=seed,
                    num_eval_runs=0,
                    horizon_days=horizon_days,
                    save_logs=False,
                    save_artifacts=False,
                )
                dm = ev.deterministic_metrics or {}
                det_records.append({
                    'checkpoint_prefix': ckpt_prefix,
                    'checkpoint_label': ckpt_label,
                    'years': int(years),
                    'horizon_days': int(horizon_days),
                    'start_offset': int(start_offset),
                    **meta,
                    'det_return': float(dm.get('total_return', np.nan)),
                    'det_sharpe': float(dm.get('sharpe_ratio', np.nan)),
                    'det_mdd': float(dm.get('max_drawdown_abs', np.nan)),
                    'det_turnover': float(dm.get('turnover', np.nan)),
                })
            except Exception as e:
                det_records.append({
                    'checkpoint_prefix': ckpt_prefix,
                    'checkpoint_label': ckpt_label,
                    'years': int(years),
                    'horizon_days': int(horizon_days),
                    'start_offset': int(start_offset),
                    'error': f'{type(e).__name__}: {e}',
                })

deterministic_sweep_df = pd.DataFrame(det_records)
if 'det_sharpe' not in deterministic_sweep_df.columns:
    raise RuntimeError('Deterministic sweep produced no successful evaluations.')

valid_det_df = deterministic_sweep_df[deterministic_sweep_df['det_sharpe'].notna()].copy()
if valid_det_df.empty:
    sample_errors = deterministic_sweep_df.get('error', pd.Series(dtype=str)).dropna().head(10).tolist()
    raise RuntimeError(f'All deterministic evaluations failed. Sample errors: {sample_errors}')

deterministic_checkpoint_summary = (
    valid_det_df
    .groupby(['checkpoint_label', 'checkpoint_prefix', 'years'], as_index=False)
    .agg(
        horizon_days=('horizon_days', 'first'),
        det_sharpe_median=('det_sharpe', 'median'),
        det_return_median=('det_return', 'median'),
        det_mdd_median=('det_mdd', 'median'),
        det_turnover_median=('det_turnover', 'median'),
    )
)
deterministic_checkpoint_summary['selection_score'] = (
    deterministic_checkpoint_summary['det_sharpe_median']
    + 0.20 * deterministic_checkpoint_summary['det_return_median']
    - 0.70 * deterministic_checkpoint_summary['det_mdd_median']
    - 0.10 * deterministic_checkpoint_summary['det_turnover_median']
)

deterministic_horizon_winners = (
    deterministic_checkpoint_summary
    .sort_values(['years', 'selection_score', 'det_sharpe_median'], ascending=[True, False, False])
    .groupby('years', as_index=False)
    .head(1)
    .reset_index(drop=True)
)

display(deterministic_horizon_winners)
save_df(deterministic_sweep_df, EVAL_DIRS['checkpoint_selection'] / 'deterministic_sweep_raw.csv')
save_df(deterministic_checkpoint_summary, EVAL_DIRS['checkpoint_selection'] / 'deterministic_checkpoint_summary.csv')
save_df(deterministic_horizon_winners, EVAL_DIRS['checkpoint_selection'] / 'deterministic_horizon_winners.csv')


## 6) Stratified Stochastic Confirmation
There are two stratified stochastic blocks:
- regime-stratified windows
- year-stratified windows

These are run on the deterministic winners, not on the full checkpoint basket.


In [ ]:
def build_regime_windows(date_series: pd.Series, horizon_days: int = 252) -> pd.DataFrame:
    dates = pd.to_datetime(pd.Series(date_series)).dropna().reset_index(drop=True)
    date_df = pd.DataFrame({"Date": dates})
    date_df["year"] = date_df["Date"].dt.year

    def _regime_label(ts: pd.Timestamp) -> str:
        if ts < pd.Timestamp("2020-03-01"):
            return "pre_covid"
        if ts <= pd.Timestamp("2020-08-31"):
            return "covid_crash"
        if ts <= pd.Timestamp("2021-12-31"):
            return "post_covid_recovery"
        if ts <= pd.Timestamp("2023-12-31"):
            return "inflation_rates"
        return "recent"

    date_df["regime"] = date_df["Date"].map(_regime_label)
    max_start = len(date_df) - horizon_days
    rows = []
    for regime_name, group in date_df.iloc[: max_start + 1].groupby("regime"):
        offsets = group.index.to_list()
        if not offsets:
            continue
        picks = sorted(set([offsets[len(offsets)//6], offsets[len(offsets)//2], offsets[(5*len(offsets))//6]])) if len(offsets) >= 6 else [offsets[len(offsets)//2]]
        for start_offset in picks:
            rows.append({"bucket": regime_name, "start_offset": int(start_offset), "horizon_days": int(horizon_days)})
    return pd.DataFrame(rows).sort_values(["bucket", "start_offset"]).reset_index(drop=True)


def build_year_windows(date_series: pd.Series, horizon_days: int = 252, windows_per_year: int = 3, min_gap_days: int = 42) -> pd.DataFrame:
    dates = pd.to_datetime(pd.Series(date_series)).dropna().reset_index(drop=True)
    max_start = len(dates) - horizon_days
    start_df = pd.DataFrame({"offset": np.arange(max_start + 1), "Date": dates.iloc[: max_start + 1].values})
    start_df["year"] = pd.to_datetime(start_df["Date"]).dt.year
    rows = []
    for year, group in start_df.groupby("year"):
        group = group.reset_index(drop=True)
        if group.empty:
            continue
        picks = np.linspace(0, len(group) - 1, num=min(windows_per_year, len(group))).round().astype(int)
        chosen = []
        for pos in sorted(set(picks.tolist())):
            row = group.iloc[pos]
            dt = pd.Timestamp(row["Date"])
            if any(abs((dt - prev).days) < min_gap_days for prev in chosen):
                continue
            chosen.append(dt)
            rows.append({"bucket": str(int(year)), "start_offset": int(row["offset"]), "horizon_days": int(horizon_days)})
    return pd.DataFrame(rows).sort_values(["bucket", "start_offset"]).reset_index(drop=True)


test_dates = pd.to_datetime(eval_phase1_data.test_df["Date"]).drop_duplicates().sort_values().reset_index(drop=True)
regime_windows_df = build_regime_windows(test_dates)
year_windows_df = build_year_windows(test_dates, windows_per_year=YEAR_WINDOWS_PER_YEAR, min_gap_days=YEAR_WINDOW_MIN_GAP_DAYS)

display(regime_windows_df)
display(year_windows_df)
save_df(regime_windows_df, EVAL_DIRS["stratified_regime"] / "regime_windows.csv")
save_df(year_windows_df, EVAL_DIRS["stratified_year"] / "year_windows.csv")


In [ ]:
def run_stratified_stochastic(winners_df: pd.DataFrame, windows_df: pd.DataFrame, output_dir: Path, label_prefix: str) -> pd.DataFrame:
    rows = []
    for _, winner in winners_df.iterrows():
        ckpt_prefix = winner["checkpoint_prefix"]
        years = int(winner["years"])
        horizon_days = int(winner["horizon_days"])
        for _, win in windows_df.iterrows():
            start_offset = int(win["start_offset"])
            phase1_slice, meta = make_phase1_slice(eval_phase1_data, start_offset, horizon_days)
            if phase1_slice is None:
                continue
            seed = int(EVAL_RANDOM_SEED + years * 10_000 + start_offset)
            try:
                ev = eval_run_one_checkpoint(
                    eval_config,
                    phase1_slice,
                    ckpt_prefix,
                    seed=seed,
                    num_eval_runs=STOCH_RUNS_PER_WINDOW,
                    horizon_days=horizon_days,
                    save_logs=True,
                    save_artifacts=True,
                )
                sto = ev.stochastic_results if isinstance(ev.stochastic_results, pd.DataFrame) else pd.DataFrame()
                if sto.empty:
                    continue
                sto = sto.copy()
                sto["bucket"] = win["bucket"]
                sto["start_offset"] = start_offset
                sto["years"] = years
                sto["checkpoint_prefix"] = ckpt_prefix
                sto["checkpoint_label"] = winner["checkpoint_label"]
                sto["window_start_date"] = meta["start_date"]
                sto["window_end_date"] = meta["end_date"]
                rows.append(sto)
            except Exception as e:
                rows.append(pd.DataFrame([{
                    "checkpoint_prefix": ckpt_prefix,
                    "checkpoint_label": winner["checkpoint_label"],
                    "years": years,
                    "bucket": win["bucket"],
                    "start_offset": start_offset,
                    "error": f"{type(e).__name__}: {e}",
                }]))
    if not rows:
        return pd.DataFrame()
    out = pd.concat(rows, ignore_index=True, sort=False)
    save_df(out, output_dir / f"{label_prefix}_stochastic_raw.csv")
    valid = out[out.get("sharpe_ratio").notna()].copy() if "sharpe_ratio" in out.columns else pd.DataFrame()
    if not valid.empty:
        summary = (
            valid
            .groupby(["checkpoint_label", "checkpoint_prefix", "years", "bucket"], as_index=False)
            .agg(
                sharpe_mean=("sharpe_ratio", "mean"),
                sharpe_std=("sharpe_ratio", "std"),
                return_mean=("total_return", "mean"),
                mdd_mean=("max_drawdown_abs", "mean"),
                turnover_mean=("turnover", "mean"),
            )
        )
        save_df(summary, output_dir / f"{label_prefix}_stochastic_summary.csv")
    capture_recent_eval_artifacts(output_dir / "captured_artifacts")
    return out

regime_stochastic_df = run_stratified_stochastic(deterministic_horizon_winners, regime_windows_df, EVAL_DIRS["stratified_regime"], "regime")
year_stochastic_df = run_stratified_stochastic(deterministic_horizon_winners, year_windows_df, EVAL_DIRS["stratified_year"], "year")

display(regime_stochastic_df.head())
display(year_stochastic_df.head())


## 7) Benchmarking vs Equal-Weight and SPY
Benchmark the deterministic horizon winners against equal-weight and SPY across the same 1, 2, 3, and 4 year horizons.


In [ ]:
def eval_identify_return_column(df: pd.DataFrame):
    for c in ['LogReturn_1d', 'log_return_1d', 'Return_1d', 'return_1d', 'daily_return']:
        if c in df.columns:
            return c
    return None


def eval_fetch_spy_returns(start_date: pd.Timestamp, end_date: pd.Timestamp) -> pd.Series:
    try:
        import yfinance as yf
    except Exception:
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'yfinance'])
            import yfinance as yf
        except Exception:
            print('[WARN] Could not install/import yfinance; SPY benchmark disabled.')
            return pd.Series(dtype=float)
    try:
        df = yf.download('SPY', start=str(start_date.date()), end=str((end_date + pd.Timedelta(days=1)).date()), auto_adjust=True, progress=False)
        if df is None or df.empty:
            return pd.Series(dtype=float)
        close = df['Close']
        if isinstance(close, pd.DataFrame):
            close = close.iloc[:, 0]
        ret = pd.to_numeric(close, errors='coerce').dropna().pct_change().dropna().astype(float)
        ret.index = pd.to_datetime(ret.index)
        return ret
    except Exception as e:
        print(f'[WARN] SPY fetch failed: {type(e).__name__}: {e}')
        return pd.Series(dtype=float)


def build_baselines_from_phase1(phase1_data: Phase1Dataset):
    test_df = phase1_data.test_df.copy()
    if 'Date' not in test_df.columns:
        raise ValueError('test_df must contain Date column')
    ret_col = eval_identify_return_column(test_df)
    if ret_col is None:
        raise ValueError('Could not identify return column in test_df')
    if 'log' in ret_col.lower():
        test_df['_simple_ret'] = np.expm1(test_df[ret_col].astype(float))
    else:
        test_df['_simple_ret'] = test_df[ret_col].astype(float)
    eqw = test_df.groupby('Date')['_simple_ret'].mean().sort_index().astype(float)
    dt_index = pd.to_datetime(eqw.index)
    spy = eval_fetch_spy_returns(dt_index.min(), dt_index.max())
    if not spy.empty:
        spy = spy.reindex(dt_index).fillna(0.0)
    return eqw.reset_index(drop=True), spy.reset_index(drop=True) if not spy.empty else pd.Series(dtype=float)


def baseline_slice(series: pd.Series, start_offset: int, horizon_days: int) -> pd.Series:
    if series is None or len(series) == 0:
        return pd.Series(dtype=float)
    end = min(len(series), int(start_offset) + int(horizon_days))
    return pd.Series(series.iloc[int(start_offset):end]).reset_index(drop=True).astype(float)


eval_baseline_eqw, eval_baseline_spy = build_baselines_from_phase1(eval_phase1_data)
benchmark_rows = []
for _, winner in deterministic_horizon_winners.iterrows():
    ckpt_prefix = winner['checkpoint_prefix']
    ckpt_label = winner['checkpoint_label']
    years = int(winner['years'])
    horizon_days = int(winner['horizon_days'])
    for start_offset in EVAL_SWEEP_START_OFFSETS:
        phase1_slice, meta = make_phase1_slice(eval_phase1_data, start_offset, horizon_days)
        if phase1_slice is None:
            continue
        ev = eval_run_one_checkpoint(
            eval_config,
            phase1_slice,
            ckpt_prefix,
            seed=int(EVAL_RANDOM_SEED + 900_000 + years * 1000 + start_offset),
            num_eval_runs=0,
            horizon_days=horizon_days,
            save_logs=True,
            save_artifacts=True,
        )
        det = ev.deterministic_metrics or {}
        row = {
            'checkpoint_label': ckpt_label,
            'checkpoint_prefix': ckpt_prefix,
            'years': years,
            'horizon_days': horizon_days,
            'start_offset': int(start_offset),
            **meta,
            'det_sharpe': float(det.get('sharpe_ratio', np.nan)),
            'det_return': float(det.get('total_return', np.nan)),
            'det_mdd': float(det.get('max_drawdown_abs', np.nan)),
            'det_turnover': float(det.get('turnover', np.nan)),
        }
        try:
            cmp_eqw = compare_agent_vs_baseline(ev, baseline_slice(eval_baseline_eqw, start_offset, horizon_days))
            for k, v in cmp_eqw.items():
                row[f'eqw_{k}'] = v
        except Exception as e:
            row['eqw_error'] = str(e)
        try:
            spy_slice = baseline_slice(eval_baseline_spy, start_offset, horizon_days)
            if len(spy_slice) > 0:
                cmp_spy = compare_agent_vs_baseline(ev, spy_slice)
                for k, v in cmp_spy.items():
                    row[f'spy_{k}'] = v
            else:
                row['spy_error'] = 'SPY baseline unavailable'
        except Exception as e:
            row['spy_error'] = str(e)
        benchmark_rows.append(row)

benchmark_horizon_raw_df = pd.DataFrame(benchmark_rows)
summary_agg = {
    'det_sharpe': 'mean',
    'det_return': 'mean',
    'det_mdd': 'mean',
    'det_turnover': 'mean',
}
rename_map = {
    'det_sharpe': 'det_sharpe_mean',
    'det_return': 'det_return_mean',
    'det_mdd': 'det_mdd_mean',
    'det_turnover': 'det_turnover_mean',
}
for col in ['eqw_agent_sharpe', 'eqw_baseline_sharpe', 'spy_agent_sharpe', 'spy_baseline_sharpe']:
    if col in benchmark_horizon_raw_df.columns:
        summary_agg[col] = 'mean'
        rename_map[col] = f'{col}_mean'

benchmark_horizon_summary_df = (
    benchmark_horizon_raw_df
    .groupby(['checkpoint_label', 'checkpoint_prefix', 'years'], as_index=False)
    .agg(summary_agg)
    .rename(columns=rename_map)
)

save_df(benchmark_horizon_raw_df, EVAL_DIRS['benchmarks'] / 'benchmark_horizon_raw.csv')
save_df(benchmark_horizon_summary_df, EVAL_DIRS['benchmarks'] / 'benchmark_horizon_summary.csv')
capture_recent_eval_artifacts(EVAL_DIRS['benchmarks'] / 'captured_artifacts')

display(benchmark_horizon_summary_df)


## 8) Export Manifest
Every analysis run should leave a simple manifest so the saved outputs are easy to navigate later.


In [ ]:
manifest = {
    'run_id': RUN_ID,
    'session_tag': SESSION_TAG,
    'results_root': str(EVAL_RESULTS_ROOT),
    'analysis_root': str(EVAL_ANALYSIS_ROOT),
    'metadata_path': str(EVAL_METADATA_PATH),
    'prep_artifacts_dir': str(EVAL_PREP_ARTIFACTS_DIR),
    'directories': {k: str(v) for k, v in EVAL_DIRS.items()},
    'horizons_days': HORIZON_DAYS,
    'deterministic_start_offsets': EVAL_SWEEP_START_OFFSETS,
    'stochastic_runs_per_window': STOCH_RUNS_PER_WINDOW,
    'manual_checkpoint_mode': USE_MANUAL_CHECKPOINT_EPISODES,
    'manual_checkpoint_episodes': MANUAL_CHECKPOINT_EPISODES if USE_MANUAL_CHECKPOINT_EPISODES else [],
    'high_watermark_subdir': EVAL_HW_SUBDIR,
    'step_sharpe_subdir': EVAL_STEP_SUBDIR,
}
save_json(manifest, EVAL_DIRS['manifests'] / 'analysis_manifest.json')
manifest
